# Azure AI Foundry model-retirement notices from Service Health

This tutorial queries the personalized Azure Service Health feed for upcoming model-retirement announcements that may affect Azure AI Foundry resources. It then turns the results into a small JSON payload suitable for a dashboard, scheduled job, or API response.

**Audience:** Azure administrators and developers who operate Foundry model deployments.

**Prerequisites**

- Python 3.10+ and an Azure subscription.
- `Reader` access (or equivalent `Microsoft.ResourceHealth/events/read` permission) on each subscription queried.
- For a live query, authenticate with `az login`, a service principal, or managed identity.

**Learning goals**

- Query `ServiceHealthResources` with the Azure Resource Graph SDK.
- Select upcoming `HealthAdvisory` / `Retirement` events related to Foundry models.
- Publish a compact, JSON-safe announcements payload.

> Service Health is personalized and subscription-scoped. It is not a global model-lifecycle catalog; the live result contains only notices visible to the signed-in identity.


## Outline

1. Install dependencies and load configuration.
2. Define the retirement query.
3. Run against a safe sample or live Azure data.
4. Shape the notices for downstream consumers.
5. Try a model watch-list exercise.


In [ ]:
%pip install -q "azure-identity>=1.17" "azure-mgmt-resourcegraph>=8.0" "pandas>=2.2" "python-dotenv>=1.0"


## 1. Configuration

Keep environment-specific settings in a `.env` file beside this notebook. Live access is off by default, so the tutorial is runnable without Azure credentials.

```dotenv
AZURE_SUBSCRIPTION_IDS=00000000-0000-0000-0000-000000000000
RUN_AZURE_QUERY=false
```

Use comma-separated IDs for more than one subscription. `DefaultAzureCredential` supports Azure CLI credentials, managed identity, and the standard service-principal environment variables. Do not save secrets in the notebook.


In [8]:
from __future__ import annotations

from dotenv import load_dotenv

load_dotenv()  # Load configuration before constructing Azure credentials or clients.

import json
import os
from collections.abc import Sequence

import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest, QueryRequestOptions
from IPython.display import display

SUBSCRIPTION_IDS = [
    value.strip()
    for value in os.getenv("AZURE_SUBSCRIPTION_IDS", "").split(",")
    if value.strip()
]
RUN_LIVE_QUERY = "true"#os.getenv("RUN_AZURE_QUERY", "false").casefold() == "true"

print(f"Mode: {'LIVE Azure query' if RUN_LIVE_QUERY else 'offline sample'}")
print(f"Configured subscriptions: {len(SUBSCRIPTION_IDS)}")


Mode: LIVE Azure query
Configured subscriptions: 1


## 2. Query upcoming Foundry model retirements

Microsoft publishes upcoming retirements as Service Health events where `EventType` is `HealthAdvisory` and `EventSubType` is `Retirement`. The extra text filter keeps notices that mention a current or former Foundry service name and model/deployment terminology.

The `ImpactMitigationTime` field represents the advisory end time for health advisories. Filtering it to the future removes expired announcements. Keep the raw `impact` object because it can contain the affected services and regions.


In [9]:
SERVICE_HEALTH_QUERY = r"""
ServiceHealthResources
| where type =~ 'Microsoft.ResourceHealth/events'
| extend
    eventType = tostring(properties.EventType),
    eventSubType = tostring(properties.EventSubType),
    status = tostring(properties.Status),
    description = tostring(properties.Title),
    trackingId = tostring(properties.TrackingId),
    summary = tostring(properties.Summary),
    priority = tostring(properties.Priority),
    impactStartTime = todatetime(tolong(properties.ImpactStartTime)),
    impactMitigationTime = todatetime(tolong(properties.ImpactMitigationTime)),
    impact = properties.Impact
| where eventType == 'HealthAdvisory' and eventSubType == 'Retirement'
| where impactMitigationTime > now()
| extend searchable = tolower(strcat(description, ' ', summary, ' ', tostring(impact)))
| where searchable contains 'azure ai foundry'
    or searchable contains 'microsoft foundry'
    or searchable contains 'azure openai'
    or searchable contains 'azure ai services'
    or searchable contains 'cognitive services'
| where searchable contains 'model' or searchable contains 'deployment'
| project trackingId, subscriptionId, status, eventType, eventSubType,
    summary, description, priority, impactStartTime, impactMitigationTime, impact
| order by impactMitigationTime asc
""".strip()

print(SERVICE_HEALTH_QUERY)


ServiceHealthResources
| where type =~ 'Microsoft.ResourceHealth/events'
| extend
    eventType = tostring(properties.EventType),
    eventSubType = tostring(properties.EventSubType),
    status = tostring(properties.Status),
    description = tostring(properties.Title),
    trackingId = tostring(properties.TrackingId),
    summary = tostring(properties.Summary),
    priority = tostring(properties.Priority),
    impactStartTime = todatetime(tolong(properties.ImpactStartTime)),
    impactMitigationTime = todatetime(tolong(properties.ImpactMitigationTime)),
    impact = properties.Impact
| where eventType == 'HealthAdvisory' and eventSubType == 'Retirement'
| where impactMitigationTime > now()
| extend searchable = tolower(strcat(description, ' ', summary, ' ', tostring(impact)))
| where searchable contains 'azure ai foundry'
    or searchable contains 'microsoft foundry'
    or searchable contains 'azure openai'
    or searchable contains 'azure ai services'
    or searchable contains '

## 3. Fetch and inspect the notices

The synthetic row below is used only when `RUN_AZURE_QUERY=false`. Set it to `true`, provide at least one subscription ID, restart the kernel, and run all cells to use the live query.


In [10]:
# Clearly synthetic data keeps the notebook runnable without an Azure tenant.
SAMPLE_ROWS = [
    {
        "trackingId": "SAMPLE-AI-RETIREMENT-001",
        "subscriptionId": "00000000-0000-0000-0000-000000000000",
        "status": "Active",
        "eventType": "HealthAdvisory",
        "eventSubType": "Retirement",
        "summary": "Migrate deployments using Sample-GPT-Model before retirement.",
        "description": "Azure OpenAI Service: Sample-GPT-Model retirement",
        "priority": "1",
        "impactStartTime": "2026-08-01T00:00:00Z",
        "impactMitigationTime": "2027-01-15T00:00:00Z",
        "impact": [
            {
                "ImpactedService": "Azure OpenAI Service",
                "ImpactedRegions": [{"ImpactedRegion": "Global"}],
            }
        ],
    }
]


def fetch_retirement_notices(subscription_ids: Sequence[str]) -> list[dict]:
    """Return upcoming Foundry model-retirement events visible to the identity."""
    if not subscription_ids:
        raise ValueError(
            "Set AZURE_SUBSCRIPTION_IDS before enabling RUN_AZURE_QUERY."
        )

    credential = DefaultAzureCredential()
    client = ResourceGraphClient(credential)
    request = QueryRequest(
        subscriptions=list(subscription_ids),
        query=SERVICE_HEALTH_QUERY,
        options=QueryRequestOptions(result_format="objectArray"),
    )

    try:
        response = client.resources(request)
        return list(response.data)
    finally:
        client.close()
        credential.close()


In [11]:
DISPLAY_COLUMNS = [
    "trackingId",
    "subscriptionId",
    "status",
    "eventType",
    "eventSubType",
    "description",
    "summary",
    "priority",
    "impactStartTime",
    "impactMitigationTime",
    "impact",
]


def as_frame(rows: Sequence[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows).reindex(columns=DISPLAY_COLUMNS)
    for column in ("impactStartTime", "impactMitigationTime"):
        frame[column] = pd.to_datetime(frame[column], errors="coerce", utc=True)
    return frame.sort_values("impactMitigationTime", na_position="last")


rows = fetch_retirement_notices(SUBSCRIPTION_IDS) if RUN_LIVE_QUERY else SAMPLE_ROWS
notices = as_frame(rows)

print(f"Found {len(notices)} matching notice(s).")
display(notices.drop(columns="impact"))


Found 2 matching notice(s).


,trackingId,subscriptionId,status,eventType,eventSubType,description,summary,priority,impactStartTime,impactMitigationTime
0,HQ3Q-XHZ,6025ba02-1dfd-407f-b358-88f811c7c7aa,Active,HealthAdvisory,Retirement,Retirement notice: Upgrade from Sora 2 2025-10...,<p><strong><em>You're receiving this notificat...,22,2026-06-03 12:04:12+00:00,2027-12-03 12:04:12+00:00
1,TJN2-21G,6025ba02-1dfd-407f-b358-88f811c7c7aa,Active,HealthAdvisory,Retirement,Retirement notice: Azure OpenAI o3-mini model ...,<p><strong><em>You're receiving this notificat...,22,2026-07-14 19:25:21+00:00,2028-01-14 19:25:21+00:00


## 4. Expose a compact JSON payload

This final transformation is intentionally transport-neutral. Return `announcement_payload` from an HTTP endpoint, write it to storage, or pass it to a notification workflow. The tracking ID and subscription ID form a useful de-duplication key.


In [12]:
PUBLIC_COLUMNS = [
    "trackingId",
    "subscriptionId",
    "status",
    "description",
    "summary",
    "priority",
    "impactStartTime",
    "impactMitigationTime",
    "impact",
]

announcement_payload = {
    "source": "Azure Service Health via Azure Resource Graph",
    "count": len(notices),
    "announcements": json.loads(
        notices[PUBLIC_COLUMNS].to_json(orient="records", date_format="iso")
    ),
}

print(json.dumps(announcement_payload, indent=2))


{
  "source": "Azure Service Health via Azure Resource Graph",
  "count": 2,
  "announcements": [
    {
      "trackingId": "HQ3Q-XHZ",
      "subscriptionId": "6025ba02-1dfd-407f-b358-88f811c7c7aa",
      "status": "Active",
      "description": "Retirement notice: Upgrade from Sora 2 2025-10-06 to Upgrade 2025-12-08 version by 15 July 2026",
      "summary": "<p><strong><em>You're receiving this notification because you're associated with one or more Azure subscriptions that use Sora 2 model version 2025-10-06.</em></strong></p>",
      "priority": "22",
      "impactStartTime": "2026-06-03T12:04:12.000Z",
      "impactMitigationTime": "2027-12-03T12:04:12.000Z",
      "impact": [
        {
          "ImpactedServiceGuid": "a8020f64-da23-440a-9f38-3e379992e725",
          "ImpactedRegions": [
            {
              "ImpactedRegion": "Australia East",
              "Status": "Active"
            },
            {
              "ImpactedRegion": "Brazil South",
              "Statu

## Exercise: add a model watch list

Filter `notices` to a set of model names your team currently deploys. Match against both the title and summary, and make the comparison case-insensitive. Predict whether the synthetic notice will remain before running the answer cell.


In [ ]:
# One possible answer
WATCHED_MODELS = ["sample-gpt-model", "gpt-4o"]


def filter_to_watched_models(
    frame: pd.DataFrame, model_names: Sequence[str]
) -> pd.DataFrame:
    terms = [name.casefold() for name in model_names]
    searchable = (
        frame[["description", "summary"]]
        .fillna("")
        .agg(" ".join, axis=1)
        .str.casefold()
    )
    mask = searchable.apply(lambda text: any(term in text for term in terms))
    return frame.loc[mask].copy()


watched_notices = filter_to_watched_models(notices, WATCHED_MODELS)
display(watched_notices[["description", "impactMitigationTime"]])


## Pitfalls and next steps

- **Use `EventSubType`, not `ImpactType`:** current retirement events are `HealthAdvisory` + `Retirement`.
- **An empty result may be valid:** confirm the identity can read the selected subscriptions and inspect all retirement advisories before assuming there are no notices.
- **Product wording changes:** periodically inspect `description`, `summary`, and `impact`, then update the Foundry terms in the query.
- **Operationalize carefully:** schedule the query, de-duplicate on `(trackingId, subscriptionId)`, and alert only on new or changed records. Add Azure Monitor/Application Insights telemetry and secret rotation before production use.

### References

- [Azure Service Health Resource Graph samples](https://learn.microsoft.com/azure/service-health/resource-graph-samples#all-upcoming-service-retirement-events)
- [Run an Azure Resource Graph query with Python](https://learn.microsoft.com/azure/governance/resource-graph/first-query-python)
- [`DefaultAzureCredential` guidance](https://learn.microsoft.com/python/api/overview/azure/identity-readme)
